## TicketSense — Helpdesk Ticket Classifier & Router

This section will demonstrate how to build a Helpdesk Ticket Classifier & Router model to classify spam messages using the `/content/dataset-tickets-german_normalized.csv` dataset.

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# Load the dataset
try:
    df = pd.read_csv('/content/dataset-tickets-german_normalized.csv', encoding='latin-1')
except FileNotFoundError:
    print("Error: spam.csv not found. Please upload the file or provide the correct path.")
    # Create a dummy DataFrame for demonstration if the file is not found
    data = {'v1': ['ham', 'spam', 'ham', 'spam', 'ham'],
            'v2': ['Go until jurong point, crazy..', 'Free entry in 2 a wkly comp to win FA Cup finals tkts 21st May 2005.', 'U dun say so early hor... U c already then say...', 'Had your mobile 11 months or more? U R entitled to Update to the latest colour mobiles with camera for Free!', 'Nah I dont think he goes to usf, he only walks around here.']}
    df = pd.DataFrame(data)
    print("Using a dummy DataFrame for demonstration purposes.")

# Display the first few rows and column information
print("Dataset Head:")
display(df.head())
print("\nDataset Info:")
df.info()

Dataset Head:


,subject,body,queue,priority,language
0,Dringend: Fehler bei der Zahlungsabwicklung de...,"Liebes Support-Team, ich kontaktiere Sie im Na...",Account & Billing Management,high,de
1,Build-Fehler aufgetreten,"Hallo Support-Team, wir haben gelegentlich Bui...",Software Development,medium,de
2,Aktualisierung zur HIPAA-KonformitÃ¤t fÃ¼r CMS,"Liebes Support-Team, ich benÃ¶tige UnterstÃ¼tz...",Legal & Compliance Requests,low,de
3,Probleme mit der HubSpot-FunktionalitÃ¤t,"Liebes Support-Team, ich habe unregelmÃ¤Ãige ...",User Experience & Design Feedback,medium,de
4,Problem mit der Telemedizin-Plattform,"Liebes Support-Team, ich habe ein schwerwiegen...",User Experience & Design Feedback,high,de



Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2125 entries, 0 to 2124
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   subject   2125 non-null   object
 1   body      2125 non-null   object
 2   queue     2125 non-null   object
 3   priority  2125 non-null   object
 4   language  2125 non-null   object
dtypes: object(5)
memory usage: 83.1+ KB


### Data Preprocessing

1.  **Rename columns**: Rename `v1` to `label` and `v2` to `text` for better readability.
2.  **Handle missing values**: Check and remove any rows with missing data.
3.  **Encode labels**: Convert categorical labels ('ham', 'spam') into numerical format (0, 1).

In [5]:
# Re-load the dataset to ensure 'queue' and 'body' columns are present,
# as a previous execution might have truncated the DataFrame.
import pandas as pd
try:
    df = pd.read_csv('/content/dataset-tickets-german_normalized.csv', encoding='latin-1')
    print("DataFrame reloaded to ensure correct columns for preprocessing.")
except FileNotFoundError:
    print("Error: dataset-tickets-german_normalized.csv not found during re-initialization in preprocessing step.")
    # Fallback to a dummy DataFrame if the file is still not found, to allow the cell to proceed.
    # This is a fallback for demonstration purposes if the file is consistently missing.
    data = {'subject': ['Dringend: Fehler'], 'body': ['textbody'], 'queue': ['Support'], 'priority': ['High'], 'language': ['German']}
    df = pd.DataFrame(data)
    print("Using a dummy DataFrame due to file not found during re-initialization.")

# Select 'queue' and 'body' and rename them for clarity
df = df[['queue', 'body']].rename(columns={'queue': 'label', 'body': 'text'})

# Check for missing values
print("Missing values before dropping:")
display(df.isnull().sum())

# Drop rows with any missing values
df.dropna(inplace=True)
print("\nMissing values after dropping:")
display(df.isnull().sum())

# Encode labels: The 'label' column (originally 'queue') will have multiple categories.
# We will use factorize to convert them to numerical format.
# For this helpdesk ticket dataset, we'll encode the different queues numerically.

df['label_id'] = df['label'].factorize()[0]

print("\nDataFrame after preprocessing:")
display(df.head())
display(df['label'].value_counts())
display(df[['label', 'label_id']].drop_duplicates().sort_values('label_id'))

DataFrame reloaded to ensure correct columns for preprocessing.
Missing values before dropping:


,0
label,0
text,0



Missing values after dropping:


,0
label,0
text,0



DataFrame after preprocessing:


,label,text,label_id
0,Account & Billing Management,"Liebes Support-Team, ich kontaktiere Sie im Na...",0
1,Software Development,"Hallo Support-Team, wir haben gelegentlich Bui...",1
2,Legal & Compliance Requests,"Liebes Support-Team, ich benÃ¶tige UnterstÃ¼tz...",2
3,User Experience & Design Feedback,"Liebes Support-Team, ich habe unregelmÃ¤Ãige ...",3
4,User Experience & Design Feedback,"Liebes Support-Team, ich habe ein schwerwiegen...",3


,count
label,
Data Analytics & Reporting,240
Partner & Vendor Coordination,236
Release Management,231
Network Infrastructure,228
Legal & Compliance Requests,220
Account & Billing Management,218
Security Operations,214
User Experience & Design Feedback,187
Customer Onboarding,181


,label,label_id
0,Account & Billing Management,0
1,Software Development,1
2,Legal & Compliance Requests,2
3,User Experience & Design Feedback,3
5,Partner & Vendor Coordination,4
6,Data Analytics & Reporting,5
7,Release Management,6
8,Security Operations,7
10,Customer Onboarding,8
21,Network Infrastructure,9


### Split Data and Feature Extraction

1.  **Split the data**: Divide the dataset into training and testing sets.
2.  **TF-IDF Vectorization**: Convert the text data into numerical feature vectors using TF-IDF (Term Frequency-Inverse Document Frequency).

In [6]:
# Split data into features (X) and target (y)
X = df['text']
y = df['label']

# Split the dataset into training and testing sets
# Adjusted test_size to 0.4 to ensure enough samples for stratification with 2 classes in a small dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.4, random_state=42, stratify=y)

print(f"Training set size: {len(X_train)}")
print(f"Testing set size: {len(X_test)}")

# Initialize TF-IDF Vectorizer
tfidf_vectorizer = TfidfVectorizer(stop_words='english', max_features=5000) # Limiting features to 5000

# Fit and transform the training data
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)

# Transform the test data
X_test_tfidf = tfidf_vectorizer.transform(X_test)

print(f"\nShape of X_train_tfidf: {X_train_tfidf.shape}")
print(f"Shape of X_test_tfidf: {X_test_tfidf.shape}")

Training set size: 1275
Testing set size: 850

Shape of X_train_tfidf: (1275, 4845)
Shape of X_test_tfidf: (850, 4845)


### TicketSense — Helpdesk Ticket Classifier & Router
Initialize and train a TicketSense — Helpdesk Ticket Classifier & Router model on the TF-IDF transformed training data.

In [8]:
# Initialize the Logistic Regression model (as intended by the imports and parameters)
model = LogisticRegression(solver='liblinear', random_state=42)

# Train the model
model.fit(X_train_tfidf, y_train)

print("Logistic Regression model (TicketSense — Helpdesk Ticket Classifier & Router) trained successfully!")

Logistic Regression model (TicketSense — Helpdesk Ticket Classifier & Router) trained successfully!


### Model Evaluation

Evaluate the trained model's performance on the test set using accuracy and a classification report.

In [9]:
# Make predictions on the test set
y_pred = model.predict(X_test_tfidf)

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
report = classification_report(y_test, y_pred)

print(f"Accuracy: {accuracy:.4f}")
print("\nClassification Report:")
print(report)

Accuracy: 0.9329

Classification Report:
                                   precision    recall  f1-score   support

     Account & Billing Management       0.95      0.90      0.92        87
              Customer Onboarding       0.98      0.88      0.93        72
       Data Analytics & Reporting       0.92      1.00      0.96        96
      Legal & Compliance Requests       0.96      0.98      0.97        88
           Network Infrastructure       0.98      0.98      0.98        91
    Partner & Vendor Coordination       0.80      0.92      0.85        95
               Release Management       0.91      0.90      0.91        92
              Security Operations       0.92      0.91      0.91        86
             Software Development       1.00      0.91      0.95        68
User Experience & Design Feedback       0.99      0.95      0.97        75

                         accuracy                           0.93       850
                        macro avg       0.94      0.93   

### Additional Model Training and Evaluation

This section will train and evaluate additional machine learning models for spam detection: Decision Tree, Support Vector Machine (SVM), and K-Nearest Neighbors (KNN).

#### 1. Decision Tree Classifier

In [10]:
from sklearn.tree import DecisionTreeClassifier

# Initialize the Decision Tree Classifier
dt_model = DecisionTreeClassifier(random_state=42)

# Train the model
dt_model.fit(X_train_tfidf, y_train)

print("Decision Tree Classifier trained successfully!")

# Make predictions on the test set
y_pred_dt = dt_model.predict(X_test_tfidf)

# Evaluate the model
accuracy_dt = accuracy_score(y_test, y_pred_dt)
report_dt = classification_report(y_test, y_pred_dt)

print(f"\nDecision Tree Accuracy: {accuracy_dt:.4f}")
print("\nDecision Tree Classification Report:")
print(report_dt)

Decision Tree Classifier trained successfully!

Decision Tree Accuracy: 0.8141

Decision Tree Classification Report:
                                   precision    recall  f1-score   support

     Account & Billing Management       0.97      0.78      0.87        87
              Customer Onboarding       0.76      0.82      0.79        72
       Data Analytics & Reporting       0.92      0.98      0.95        96
      Legal & Compliance Requests       0.92      0.92      0.92        88
           Network Infrastructure       0.92      0.90      0.91        91
    Partner & Vendor Coordination       0.66      0.71      0.68        95
               Release Management       0.71      0.76      0.73        92
              Security Operations       0.66      0.63      0.64        86
             Software Development       0.89      0.97      0.93        68
User Experience & Design Feedback       0.76      0.68      0.72        75

                         accuracy                       

#### 2. Support Vector Machine (SVM)

In [11]:
from sklearn.svm import SVC

# Initialize the SVM model
svm_model = SVC(kernel='linear', random_state=42) # Using a linear kernel for text data often works well

# Train the model
svm_model.fit(X_train_tfidf, y_train)

print("SVM model trained successfully!")

# Make predictions on the test set
y_pred_svm = svm_model.predict(X_test_tfidf)

# Evaluate the model
accuracy_svm = accuracy_score(y_test, y_pred_svm)
report_svm = classification_report(y_test, y_pred_svm)

print(f"\nSVM Accuracy: {accuracy_svm:.4f}")
print("\nSVM Classification Report:")
print(report_svm)

SVM model trained successfully!

SVM Accuracy: 0.9624

SVM Classification Report:
                                   precision    recall  f1-score   support

     Account & Billing Management       0.98      0.93      0.95        87
              Customer Onboarding       0.94      0.94      0.94        72
       Data Analytics & Reporting       1.00      1.00      1.00        96
      Legal & Compliance Requests       0.99      0.97      0.98        88
           Network Infrastructure       0.99      0.99      0.99        91
    Partner & Vendor Coordination       0.88      0.94      0.91        95
               Release Management       0.95      0.97      0.96        92
              Security Operations       0.92      0.98      0.95        86
             Software Development       1.00      0.93      0.96        68
User Experience & Design Feedback       1.00      0.97      0.99        75

                         accuracy                           0.96       850
                

#### 3. K-Nearest Neighbors (KNN)

In [12]:
from sklearn.neighbors import KNeighborsClassifier

# Initialize the KNN model
# Adjusted n_neighbors to be <= n_samples_fit (3 in this case) due to small dataset
knn_model = KNeighborsClassifier(n_neighbors=3) # You can adjust n_neighbors

# Train the model
knn_model.fit(X_train_tfidf, y_train)

print("KNN model trained successfully!")

# Make predictions on the test set
y_pred_knn = knn_model.predict(X_test_tfidf)

# Evaluate the model
accuracy_knn = accuracy_score(y_test, y_pred_knn)
report_knn = classification_report(y_test, y_pred_knn)

print(f"\nKNN Accuracy: {accuracy_knn:.4f}")
print("\nKNN Classification Report:")
print(report_knn)

KNN model trained successfully!

KNN Accuracy: 0.8494

KNN Classification Report:
                                   precision    recall  f1-score   support

     Account & Billing Management       0.89      0.90      0.89        87
              Customer Onboarding       0.83      0.89      0.86        72
       Data Analytics & Reporting       0.72      0.92      0.81        96
      Legal & Compliance Requests       0.83      0.89      0.86        88
           Network Infrastructure       0.87      0.89      0.88        91
    Partner & Vendor Coordination       0.83      0.71      0.76        95
               Release Management       0.85      0.84      0.84        92
              Security Operations       0.93      0.78      0.85        86
             Software Development       0.89      0.93      0.91        68
User Experience & Design Feedback       0.97      0.79      0.87        75

                         accuracy                           0.85       850
                

### Loading a Movie Dataset

Given the difficulties with cross-validation on the extremely small dummy spam dataset, let's proceed with a new dataset: a movie dataset. This section will handle loading the movie data and displaying its initial structure.

In [16]:
import pandas as pd

# Attempt to load a movie dataset from Google Drive
# You might need to adjust the path if your movie dataset has a different name or location.
try:
    movie_df = pd.read_csv('/content/tmdb_5000_movies.csv') # Common name, adjust if needed
    print("Movie dataset loaded successfully!")
except FileNotFoundError:
    print("Error: movies.csv not found in MyDrive. Please check the path or filename.")
    print("If your movie dataset has a different name or is in a different folder, please update the path above.")
    # As a placeholder, creating a small dummy movie DataFrame if file not found
    data_placeholder = {
        'title': ['Movie A', 'Movie B', 'Movie C', 'Movie D', 'Movie E'],
        'genre': ['Action', 'Comedy', 'Drama', 'Action', 'Sci-Fi'],
        'rating': [7.5, 6.2, 8.9, 7.8, 9.1],
        'year': [2000, 2010, 2005, 2015, 1999]
    }
    movie_df = pd.DataFrame(data_placeholder)
    print("Using a dummy movie DataFrame for demonstration.")

# Display the first few rows and column information of the movie dataset
print("\nMovie Dataset Head:")
display(movie_df.head())
print("\nMovie Dataset Info:")
movie_df.info()

Movie dataset loaded successfully!

Movie Dataset Head:


,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2009-12-10,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800
1,300000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",http://disney.go.com/disneypictures/pirates/,285,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...",en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2007-05-19,961000000,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500
2,245000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.sonypictures.com/movies/spectre/,206647,"[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...",en,Spectre,A cryptic message from Bond’s past sends him o...,107.376788,"[{""name"": ""Columbia Pictures"", ""id"": 5}, {""nam...","[{""iso_3166_1"": ""GB"", ""name"": ""United Kingdom""...",2015-10-26,880674609,148.0,"[{""iso_639_1"": ""fr"", ""name"": ""Fran\u00e7ais""},...",Released,A Plan No One Escapes,Spectre,6.3,4466
3,250000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""nam...",http://www.thedarkknightrises.com/,49026,"[{""id"": 849, ""name"": ""dc comics""}, {""id"": 853,...",en,The Dark Knight Rises,Following the death of District Attorney Harve...,112.312950,"[{""name"": ""Legendary Pictures"", ""id"": 923}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2012-07-16,1084939099,165.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,The Legend Ends,The Dark Knight Rises,7.6,9106
4,260000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://movies.disney.com/john-carter,49529,"[{""id"": 818, ""name"": ""based on novel""}, {""id"":...",en,John Carter,"John Carter is a war-weary, former military ca...",43.926995,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}]","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2012-03-07,284139100,132.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"Lost in our world, found in another.",John Carter,6.1,2124



Movie Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4803 entries, 0 to 4802
Data columns (total 20 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   budget                4803 non-null   int64  
 1   genres                4803 non-null   object 
 2   homepage              1712 non-null   object 
 3   id                    4803 non-null   int64  
 4   keywords              4803 non-null   object 
 5   original_language     4803 non-null   object 
 6   original_title        4803 non-null   object 
 7   overview              4800 non-null   object 
 8   popularity            4803 non-null   float64
 9   production_companies  4803 non-null   object 
 10  production_countries  4803 non-null   object 
 11  release_date          4802 non-null   object 
 12  revenue               4803 non-null   int64  
 13  runtime               4801 non-null   float64
 14  spoken_languages      4803 non-null   object 
 15  

### Preparing Movie Data for GridSearchCV

To perform `GridSearchCV`, we need to define features (X) and a target variable (y). For this demonstration, we'll try to predict the `genre` based on `rating` and `year`. Since `genre` is categorical, we'll encode it numerically.

In [17]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report

# Handle missing values in 'genres' and 'release_date' before processing
movie_df.dropna(subset=['genres', 'release_date', 'vote_average'], inplace=True)

# Encode 'genres' column to numerical labels
le = LabelEncoder()
movie_df['genre_encoded'] = le.fit_transform(movie_df['genres'])

# Extract year from 'release_date'
movie_df['release_year'] = pd.to_datetime(movie_df['release_date']).dt.year

# Define features (X) and target (y)
X_movie = movie_df[['vote_average', 'release_year']]
y_movie = movie_df['genre_encoded']

# Split the movie dataset into training and testing sets
# We will try to stratify, but due to many unique genres, some classes may still have few samples.
# This dataset is much larger, so stratification should generally work better than with the dummy data.
# Temporarily setting stratify=None to bypass the error caused by single-sample classes.
# For a more robust solution, rare classes in y_movie would need to be handled (e.g., filtered or grouped).
X_train_movie, X_test_movie, y_train_movie, y_test_movie = train_test_split(
    X_movie, y_movie, test_size=0.2, random_state=42, stratify=None # Changed to None to resolve ValueError
)

print(f"Movie Training set size: {len(X_train_movie)}")
print(f"Movie Testing set size: {len(X_test_movie)}")
print(f"\ny_train_movie value counts (top 10):\n{y_train_movie.value_counts().head(10)}")
print(f"y_test_movie value counts (top 10):\n{y_test_movie.value_counts().head(10)}")

Movie Training set size: 3841
Movie Testing set size: 961

y_train_movie value counts (top 10):
genre_encoded
543     301
944     229
392     142
899     119
833     116
876      84
578      66
590      54
519      53
1174     48
Name: count, dtype: int64
y_test_movie value counts (top 10):
genre_encoded
543     69
944     53
833     28
876     25
899     23
392     22
578     22
1174    20
798     11
1053    11
Name: count, dtype: int64


### Running GridSearchCV on Movie Dataset (with anticipated limitations)

Now, let's attempt to run `GridSearchCV` on this prepared movie dataset. We'll use a `DecisionTreeClassifier` as an example and define a small parameter grid. We anticipate that `GridSearchCV` will encounter similar `n_splits` errors due to the extremely small number of samples per class in the training data.

In [18]:
# Define the model
dt_model_movie = DecisionTreeClassifier(random_state=42)

# Define a simple parameter grid
param_grid_movie = {
    'max_depth': [None, 2, 3],
    'min_samples_split': [2, 3]
}

# Initialize GridSearchCV
# We will use cv=2, but it's highly likely to fail because the training data (4 samples)
# has classes with only one instance (e.g., genre_encoded 0, 1, 2, 3 may each have 1 sample in y_train_movie).
# Sklearn's GridSearchCV with default StratifiedKFold for classification will struggle.
# For a real dataset, you would use a higher, meaningful cv value (e.g., 5 or 10).

try:
    grid_search_movie = GridSearchCV(dt_model_movie, param_grid_movie, cv=2, scoring='accuracy', n_jobs=-1)
    grid_search_movie.fit(X_train_movie, y_train_movie)

    print("GridSearchCV completed successfully!")
    print(f"Best parameters: {grid_search_movie.best_params_}")
    print(f"Best cross-validation score: {grid_search_movie.best_score_:.4f}")

    # Evaluate on the test set
    y_pred_grid_movie = grid_search_movie.predict(X_test_movie)
    print("\nClassification Report on Test Set:")
    print(classification_report(y_test_movie, y_pred_grid_movie))

except Exception as e:
    print(f"An error occurred during GridSearchCV: {e}")
    print("This is expected for such a small and imbalanced dummy dataset.")
    print("Specifically, with only 4 training samples and 4 unique genres,")
    print("it's impossible to create 2 folds where each class is sufficiently represented.")
    print("For a proper GridSearchCV, a larger and more balanced dataset is required.")

/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=2.
  warnings.warn(


GridSearchCV completed successfully!
Best parameters: {'max_depth': 2, 'min_samples_split': 2}
Best cross-validation score: 0.1002

Classification Report on Test Set:
              precision    recall  f1-score   support

           0       0.23      0.75      0.35         4
           1       0.00      0.00      0.00         1
           4       0.00      0.00      0.00         1
           6       0.00      0.00      0.00         1
           7       0.00      0.00      0.00         1
           8       0.00      0.00      0.00         2
          11       0.00      0.00      0.00         1
          12       0.00      0.00      0.00         2
          19       0.00      0.00      0.00         1
          28       0.00      0.00      0.00         2
          33       0.00      0.00      0.00         2
          34       0.00      0.00      0.00         4
          41       0.00      0.00      0.00         1
          46       0.00      0.00      0.00         1
          47       0.0

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
